Hi.

This notebook shows how to merge tinker weights with gpt-oss-120b model.

The idea is based on @[huikang](https://www.kaggle.com/huikang) work.

Check his work here :
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3/discussion/685037#3429493

Here is his scripts:
https://github.com/tonghuikang/aimo3/blob/master/merge_adapter.py

Which he used in his corpus prize submission :
https://www.kaggle.com/competitions/ai-mathematical-olympiad-progress-prize-3/discussion/672528

Please note this code worked for me (from merging to uploading to Kaggle), I used a colab with 225.3 GB hard disk.
The code might need some cleaning.

I didn't test on Kaggle (I don't think there is enough hard disk space for that)

Use version 2 , which should be more stable when it comes to space.

In [1]:
# ==========================================
# 1) Install
# ==========================================
!pip -q install -U kagglehub safetensors numba psutil
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu130

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 9.2 MB/s eta 0:00:00
Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 266.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 350.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 312.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 MB 163.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 155.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 185.7 MB/

In [2]:
!pip install --no-cache-dir --force-reinstall "tinker-cookbook @ git+https://github.com/thinking-machines-lab/tinker-cookbook.git@nightly"

  Cloning https://github.com/thinking-machines-lab/tinker-cookbook.git (to revision nightly) to /tmp/pip-install-yuewgbbm/tinker-cookbook_9a547ce81e8843159577540a0d9a3311
  Running command git clone --filter=blob:none --quiet https://github.com/thinking-machines-lab/tinker-cookbook.git /tmp/pip-install-yuewgbbm/tinker-cookbook_9a547ce81e8843159577540a0d9a3311
  Resolved https://github.com/thinking-machines-lab/tinker-cookbook.git to commit 5f4219e1f8460641a5c84cb2c05fc7f98899cf22
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 35.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 443.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 489.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 452.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 k

In [3]:
from tinker_cookbook import weights

ModuleNotFoundError: Could not import module 'AutoProcessor'. Are this object's requirements defined correctly?

In [4]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [8]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="unsloth/gpt-oss-120b" , local_dir="base" , ignore_patterns=["original/*", "metal/*"])


Fetching 27 files:   0%|          | 0/27 [00:00<?, ?it/s]

'/content/base'

In [9]:
import os
os.listdir("base")

['model-00004-of-00014.safetensors',
 '.gitattributes',
 'model-00006-of-00014.safetensors',
 'model-00013-of-00014.safetensors',
 'tokenizer_config.json',
 'model-00014-of-00014.safetensors',
 'chat_template.json',
 'model-00008-of-00014.safetensors',
 'model-00012-of-00014.safetensors',
 'model-00001-of-00014.safetensors',
 'model-00007-of-00014.safetensors',
 'chat_template.jinja',
 'model-00011-of-00014.safetensors',
 'model-00002-of-00014.safetensors',
 'USAGE_POLICY',
 'model-00005-of-00014.safetensors',
 'LICENSE',
 'tokenizer.json',
 'model-00009-of-00014.safetensors',
 'README.md',
 'model-00010-of-00014.safetensors',
 'generation_config.json',
 'model-00003-of-00014.safetensors',
 'config.json',
 'model.safetensors.index.json',
 '.cache',
 'model-00000-of-00014.safetensors',
 'special_tokens_map.json']

In [ ]:

import os
import gc
import json
import time
import shutil
import tarfile
import zipfile
from pathlib import Path
from dataclasses import dataclass

import kagglehub
import numpy as np
import psutil
import torch
from numba import njit, prange
from safetensors import safe_open
from safetensors.torch import load_file, save_file


# ============================================================
# USER CONFIG
# ============================================================

BASE_MODEL_HANDLE = "unsloth/gpt-oss-120b"
ADAPTER_DATASET_HANDLE = "barnobarno/tinker-no-tools-10k"

UPLOAD_AS = "model"   # "model" or "dataset"

UPLOAD_MODEL_HANDLE = "barnobarno/gpt-oss-120b-merged-4/Transformers/default"
UPLOAD_DATASET_HANDLE = "barnobarno/gpt-oss-120b-merged-4"

VERSION_NOTES = "Merged gpt-oss-120b with LoRA adapter on CPU, shard-by-shard."
LICENSE_NAME = None


# ============================================================
# PATHS
# ============================================================

WORK = Path("/content/gptoss_merge_cpu")
OUT_DIR = WORK / "merged"
PEFT_OUT = WORK / "peft_adapter_new"
KGH_CACHE = Path("/content/kagglehub_cache")

CLEAN_ON_START = True


# ============================================================
# AUTH
# ============================================================


# ============================================================
# HELPERS
# ============================================================

def log(*x):
    print(time.strftime("[%Y-%m-%d %H:%M:%S]"), *x)

def gb(n):
    return n / (1024 ** 3)

def safe_rmtree(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def disk_report():
    print("\n==== DISK REPORT ====")
    seen = set()
    for p in [
        "/content",
        str(WORK),
        str(OUT_DIR),
        str(PEFT_OUT),
        str(KGH_CACHE),
        os.path.expanduser("~/.cache/kagglehub"),
    ]:
        if not p or p in seen:
            continue
        seen.add(p)
        try:
            total, used, free = shutil.disk_usage(p)
            print(f"{p}: used={gb(used):.2f} GB | free={gb(free):.2f} GB")
        except Exception:
            pass
    print("=====================\n")

def cleanup_start():
    if CLEAN_ON_START:
        log("Cleaning previous run folders...")
        for p in [WORK, KGH_CACHE]:
            safe_rmtree(p)

    ensure_dir(WORK)
    ensure_dir(OUT_DIR)
    ensure_dir(KGH_CACHE)

def find_model_root(root: Path) -> Path:
    candidates = set()

    for p in root.rglob("model.safetensors.index.json"):
        candidates.add(p.parent)
    for p in root.rglob("*.safetensors"):
        candidates.add(p.parent)

    scored = []
    for c in candidates:
        n = len(list(c.glob("*.safetensors")))
        if n:
            scored.append((n, c))

    if not scored:
        raise FileNotFoundError(f"No model shards found under {root}")

    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[0][1]

def extract_archives_if_needed(root: Path):
    extracted_any = False
    for p in list(root.rglob("*")):
        if p.is_file() and p.suffix.lower() == ".zip":
            dst = p.with_suffix("")
            ensure_dir(dst)
            with zipfile.ZipFile(p, "r") as zf:
                zf.extractall(dst)
            extracted_any = True
        elif p.is_file() and p.suffix.lower() in [".tar", ".gz", ".tgz"]:
            dst = p.parent / (p.stem + "_extracted")
            ensure_dir(dst)
            with tarfile.open(p, "r:*") as tf:
                tf.extractall(dst)
            extracted_any = True
    return extracted_any

def find_adapter_root(root: Path) -> Path:
    for _ in range(3):
        for p in root.rglob("adapter_config.json"):
            if (p.parent / "adapter_model.safetensors").exists():
                return p.parent
        if not extract_archives_if_needed(root):
            break
    raise FileNotFoundError(
        f"Could not find adapter_config.json + adapter_model.safetensors under {root}"
    )

def copy_if_exists(src_dir: Path, dst_dir: Path, names):
    for name in names:
        src = src_dir / name
        if src.exists() and src.is_file():
            shutil.copy2(src, dst_dir / name)

def estimate_sizes(model_root: Path):
    shard_files = sorted(model_root.glob("*.safetensors"))
    base_bytes = sum(p.stat().st_size for p in shard_files)
    free_bytes = shutil.disk_usage("/content").free
    log(f"Base shards: {len(shard_files)} files, {gb(base_bytes):.2f} GB")
    log(f"Free disk: {gb(free_bytes):.2f} GB")
    log(f"Approx needed before merge (base + merged + overhead): ~{gb(base_bytes * 2.05):.2f} GB")
    log(f"System RAM: {gb(psutil.virtual_memory().total):.2f} GB")

def write_merge_report(path: Path, report: dict):
    (path / "merge_report.json").write_text(json.dumps(report, indent=2))

def write_readme(path: Path):
    text = f"""Merged GPT-OSS model

Base model handle:
{BASE_MODEL_HANDLE}

Adapter dataset handle:
{ADAPTER_DATASET_HANDLE}

Notes:
{VERSION_NOTES}
"""
    (path / "README.md").write_text(text)

def upload_result(local_dir: Path):
    log("Upload starting...")
    if UPLOAD_AS.lower() == "model":
        kwargs = {}
        if LICENSE_NAME:
            kwargs["license_name"] = LICENSE_NAME
        kagglehub.model_upload(
            UPLOAD_MODEL_HANDLE,
            str(local_dir),
            version_notes=VERSION_NOTES,
            **kwargs,
        )
    elif UPLOAD_AS.lower() == "dataset":
        kagglehub.dataset_upload(
            UPLOAD_DATASET_HANDLE,
            str(local_dir),
            version_notes=VERSION_NOTES,
        )
    else:
        raise ValueError("UPLOAD_AS must be 'model' or 'dataset'")
    log("Upload finished.")


# ============================================================
# GPT-OSS MXFP4 HELPERS
# ============================================================

FP4_VALUES = np.array(
    [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0,
     -0.0, -0.5, -1.0, -1.5, -2.0, -3.0, -4.0, -6.0],
    dtype=np.float32,
)

FP4_BOUNDARIES = np.array([0.25, 0.75, 1.25, 1.75, 2.5, 3.5, 5.0], dtype=np.float32)
NUM_LAYERS = 36

@dataclass
class LoraInfo:
    lora_A: torch.Tensor
    lora_B: torch.Tensor
    scaling: float

@njit(parallel=True, cache=True, fastmath=True)
def dequantize_numba(blocks, scales, fp4_values):
    n_exp, out_dim, n_scales, _ = blocks.shape
    result = np.empty((n_exp, out_dim, n_scales * 32), dtype=np.float32)
    for e in prange(n_exp):
        for o in range(out_dim):
            for s in range(n_scales):
                scale = 2.0 ** (int(scales[e, o, s]) - 127)
                base = s * 32
                for p in range(16):
                    b = blocks[e, o, s, p]
                    result[e, o, base + p * 2] = fp4_values[b & 0x0F] * scale
                    result[e, o, base + p * 2 + 1] = fp4_values[b >> 4] * scale
    return result

@njit(parallel=True, cache=True, fastmath=True)
def quantize_numba(tensor, n_scales, boundaries):
    n_exp, out_dim, _ = tensor.shape
    blocks = np.empty((n_exp, out_dim, n_scales, 16), dtype=np.uint8)
    scales = np.empty((n_exp, out_dim, n_scales), dtype=np.uint8)

    for e in prange(n_exp):
        for o in range(out_dim):
            for s in range(n_scales):
                base = s * 32

                abs_max = 0.0
                for i in range(32):
                    v = tensor[e, o, base + i]
                    av = v if v >= 0 else -v
                    if av > abs_max:
                        abs_max = av

                if abs_max < 1e-12:
                    exp = -127
                else:
                    exp = int(np.ceil(np.log2(abs_max / 6.0)))
                    exp = max(-127, min(127, exp))

                scale_val = 2.0 ** exp
                scales[e, o, s] = np.uint8(exp + 127)

                for p in range(16):
                    n0 = tensor[e, o, base + p * 2] / scale_val
                    n1 = tensor[e, o, base + p * 2 + 1] / scale_val
                    a0 = n0 if n0 >= 0 else -n0
                    a1 = n1 if n1 >= 0 else -n1

                    q0 = q1 = 0
                    for b in range(7):
                        if a0 >= boundaries[b]:
                            q0 = b + 1
                        if a1 >= boundaries[b]:
                            q1 = b + 1

                    if n0 < 0:
                        q0 += 8
                    if n1 < 0:
                        q1 += 8

                    blocks[e, o, s, p] = np.uint8(q0 | (q1 << 4))

    return blocks, scales

def dequantize_mxfp4(blocks: torch.Tensor, scales: torch.Tensor) -> torch.Tensor:
    return torch.from_numpy(dequantize_numba(blocks.numpy(), scales.numpy(), FP4_VALUES))

def quantize_mxfp4(tensor: torch.Tensor, blocks_shape: tuple):
    blocks, scales = quantize_numba(tensor.numpy(), blocks_shape[2], FP4_BOUNDARIES)
    return torch.from_numpy(blocks), torch.from_numpy(scales)


# ============================================================
# LORA MAPPING / MERGE LOGIC
# ============================================================

def build_lora_mappings(weights_dict: dict, config: dict):
    scaling = config["lora_alpha"] / config["r"]
    bf16_map = {}
    mxfp4_map = {}

    for name in weights_dict:
        if ".lora_A.weight" not in name:
            continue

        b_name = name.replace(".lora_A.weight", ".lora_B.weight")
        if b_name not in weights_dict:
            continue

        lora_A = weights_dict[name].cpu()
        lora_B = weights_dict[b_name].cpu()

        base = name.replace(".lora_A.weight", "").replace("base_model.model.", "")
        base = base.replace("model.unembed_tokens", "lm_head").replace(".attn.", ".self_attn.")

        info = LoraInfo(lora_A=lora_A, lora_B=lora_B, scaling=scaling)

        if ".mlp.experts." in base:
            base = (
                base.replace(".w1", ".gate_proj")
                    .replace(".w2", ".down_proj")
                    .replace(".w3", ".up_proj")
            )
            mxfp4_map[base] = info
        else:
            bf16_map[base + ".weight"] = info

    return bf16_map, mxfp4_map

def apply_lora_delta(base: torch.Tensor, lora_A: torch.Tensor, lora_B: torch.Tensor, scaling: float, is_batched: bool = False):
    A = lora_A.float()
    B = lora_B.float()

    if is_batched:
        n = base.shape[0]
        delta = torch.bmm(B.expand(n, -1, -1) * scaling, A.expand(n, -1, -1))
    else:
        delta = torch.nn.functional.linear(A.T, B * scaling).T

    return base.float() + delta

def merge_bf16(base: torch.Tensor, lora: LoraInfo):
    return apply_lora_delta(base, lora.lora_A, lora.lora_B, lora.scaling, False).to(base.dtype)

def merge_mxfp4(blocks: torch.Tensor, scales: torch.Tensor, lora: LoraInfo):
    original = dequantize_mxfp4(blocks, scales)
    merged = apply_lora_delta(original, lora.lora_A, lora.lora_B, lora.scaling, True)
    return quantize_mxfp4(merged.contiguous(), blocks.shape)

def merge_fused_gate_up(blocks: torch.Tensor, scales: torch.Tensor, gate_lora: LoraInfo, up_lora: LoraInfo):
    """
    BUG FIX: GPT-OSS-120B uses INTERLEAVED layout for gate_up_proj,
    NOT first-half/second-half.
    See modeling_gpt_oss.py _apply_gate():
        gate, up = gate_up[..., ::2], gate_up[..., 1::2]
    Even rows (0,2,4,...) = gate (w1), Odd rows (1,3,5,...) = up (w3).
    """
    original = dequantize_mxfp4(blocks, scales)
    n_exp, fused_dim, in_dim = original.shape
    half = fused_dim // 2
    merged = original.clone()

    merged[:, 0::2, :] += apply_lora_delta(
        torch.zeros(n_exp, half, in_dim),
        gate_lora.lora_A,
        gate_lora.lora_B,
        gate_lora.scaling,
        True,
    )
    merged[:, 1::2, :] += apply_lora_delta(
        torch.zeros(n_exp, half, in_dim),
        up_lora.lora_A,
        up_lora.lora_B,
        up_lora.scaling,
        True,
    )

    return quantize_mxfp4(merged.contiguous(), blocks.shape)


# ============================================================
# MAIN
# ============================================================

def main():
    cleanup_start()
    os.environ["KAGGLEHUB_CACHE"] = str(KGH_CACHE)

    disk_report()

# Download a model repository

    # 1) Download base model ONCE from Kaggle
    log("Downloading base model from Kaggle Models...")
    base_download = "base" #hugging Path(kagglehub.model_download(BASE_MODEL_HANDLE))
    model_root ="base" # find_model_root(base_download)
    log("Base model root:", model_root)

    disk_report()

    # 2) Download adapter dataset
    log("Downloading adapter dataset from Kaggle...")
    adapter_download = Path(kagglehub.dataset_download(ADAPTER_DATASET_HANDLE))
    raw_adapter_root = find_adapter_root(adapter_download)
    log("Raw adapter root:", raw_adapter_root)

    disk_report()

    # 3) Convert adapter using the LOCAL Kaggle model path
    #    PEFT_OUT must NOT already exist
    PEFT_OUT.parent.mkdir(parents=True, exist_ok=True)
    safe_rmtree(PEFT_OUT)

    log("Running weights.build_lora_adapter(...) using local Kaggle model path...")
    weights.build_lora_adapter(
        base_model=str(model_root),   # <-- local path, no HF download
        adapter_path=raw_adapter_root,
        output_path=str(PEFT_OUT),
    )

    adapter_root = PEFT_OUT
    log("Built PEFT adapter at:", adapter_root)

    estimate_sizes(model_root)
    disk_report()

    # 4) Load converted adapter
    with open(adapter_root / "adapter_config.json", "r") as f:
        adapter_config = json.load(f)

    adapter_weights = load_file(str(adapter_root / "adapter_model.safetensors"), device="cpu")
    bf16_map, mxfp4_map = build_lora_mappings(adapter_weights, adapter_config)

    log("Adapter tensors:", len(adapter_weights))
    log("LoRA mappings:", len(bf16_map), "BF16,", len(mxfp4_map), "MXFP4")

    # 5) Merge shard-by-shard on CPU
    shard_files = sorted(model_root.glob("*.safetensors"))
    if not shard_files:
        raise FileNotFoundError(f"No .safetensors shards found in {model_root}")

    merged_keys = set()

    for i, shard_file in enumerate(shard_files, 1):
        log(f"Shard {i}/{len(shard_files)}: {shard_file.name}")

        with safe_open(str(shard_file), framework="pt", device="cpu") as f:
            tensors = {k: f.get_tensor(k) for k in f.keys()}

        merged = {}

        for key, tensor in tensors.items():
            if key in bf16_map:
                merged[key] = merge_bf16(tensor, bf16_map[key])
                merged_keys.add(key)
                log("  BF16 merged:", key)

        for layer in range(NUM_LAYERS):
            prefix = f"model.layers.{layer}.mlp.experts"

            blocks_key = f"{prefix}.down_proj_blocks"
            scales_key = f"{prefix}.down_proj_scales"
            lora_key = f"{prefix}.down_proj"
            if blocks_key in tensors and scales_key in tensors and lora_key in mxfp4_map:
                mb, ms = merge_mxfp4(
                    tensors[blocks_key],
                    tensors[scales_key],
                    mxfp4_map[lora_key],
                )
                merged[blocks_key] = mb
                merged[scales_key] = ms
                merged_keys.add(lora_key)
                log("  MXFP4 merged:", lora_key)

            blocks_key = f"{prefix}.gate_up_proj_blocks"
            scales_key = f"{prefix}.gate_up_proj_scales"
            gate_key = f"{prefix}.gate_proj"
            up_key = f"{prefix}.up_proj"

            if (
                blocks_key in tensors
                and scales_key in tensors
                and gate_key in mxfp4_map
                and up_key in mxfp4_map
            ):
                mb, ms = merge_fused_gate_up(
                    tensors[blocks_key],
                    tensors[scales_key],
                    mxfp4_map[gate_key],
                    mxfp4_map[up_key],
                )
                merged[blocks_key] = mb
                merged[scales_key] = ms
                merged_keys.update([gate_key, up_key])
                log("  MXFP4 merged fused:", f"layer {layer} gate_proj + up_proj")

        for key, tensor in tensors.items():
            if key not in merged:
                merged[key] = tensor

        save_file(merged, str(OUT_DIR / shard_file.name))

        del tensors, merged
        gc.collect()

    expected = set(bf16_map.keys()) | set(mxfp4_map.keys())
    missing = sorted(expected - merged_keys)

    report = {
        "base_model_handle": BASE_MODEL_HANDLE,
        "adapter_dataset_handle": ADAPTER_DATASET_HANDLE,
        "num_bf16_targets": len(bf16_map),
        "num_mxfp4_targets": len(mxfp4_map),
        "num_merged_targets": len(merged_keys),
        "num_missing_targets": len(missing),
        "missing_targets_preview": missing[:200],
    }

    write_merge_report(OUT_DIR, report)
    write_readme(OUT_DIR)

    copy_if_exists(model_root, OUT_DIR, [
        "config.json",
        "generation_config.json",
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "model.safetensors.index.json",
        "chat_template.jinja",
        "preprocessor_config.json",
        "added_tokens.json",
    ])

    disk_report()

    # 6) Upload
    upload_result(OUT_DIR)
    log("Done.")

if __name__ == "__main__":
    main()